# Prospective Customer Prediction

Predicts whether a bank customer will subscribe to a term deposit based on demographic and campaign data.

**Best model:** Random Forest — 91% accuracy, AUC 0.73

> All logic lives in `src/`. This notebook is an interactive walkthrough — run `python3 src/train.py` for the full pipeline without Jupyter.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.config import DATA_PATH

## 1. Load and Explore Raw Data

In [ ]:
df_raw = pd.read_csv(f'../{DATA_PATH}')
print(df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

## 2. Exploratory Data Analysis

In [ ]:
from src.eda import (
    plot_target_distribution,
    plot_numeric_distributions,
    plot_outlier_boxplots,
    plot_correlation_heatmap,
    plot_categorical_vs_target,
)
from src.config import SCALE_COLS

plot_target_distribution(df_raw)

In [ ]:
plot_numeric_distributions(df_raw, SCALE_COLS)

In [ ]:
plot_outlier_boxplots(df_raw, SCALE_COLS)

## 3. Preprocessing

Steps: column standardisation → education consolidation → pdays → prev_c → deduplicate → outlier capping → encoding → feature selection → StandardScaler → train/test split.

In [ ]:
from src.preprocess import load_and_preprocess

X_train, X_test, y_train, y_test, scaler = load_and_preprocess(f'../{DATA_PATH}', save_scaler=True)
print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)
print('Class balance:\n', y_train.value_counts())

## 4. Feature Selection

Mutual information scores identify the 15 most informative features. Low-importance features are dropped (see `config.LOW_INFO_FEATURES`).

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from src.eda import plot_feature_importance
import numpy as np

mi_scores = mutual_info_classif(X_train, y_train)
plot_feature_importance(mi_scores, list(X_train.columns))

## 5. Baseline Model Comparison

All 5 models trained with default parameters; cross-validated on training set.

In [ ]:
from src.train import run_baseline_comparison

results = run_baseline_comparison(X_train, X_test, y_train, y_test)
results

## 6. Hyperparameter Tuning

**Random Forest** → RandomizedSearchCV (100 iterations, 5-fold CV)

**Logistic Regression** → GridSearchCV (C grid, l1/l2 penalty)

In [ ]:
from src.train import train_random_forest, train_logistic_regression

rf = train_random_forest(X_train, X_test, y_train, y_test, tune=True)
lr = train_logistic_regression(X_train, X_test, y_train, y_test, tune=True)

## 7. ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
from src.eda import plot_roc_curve

fpr, tpr, _ = roc_curve(y_test, rf.predict_proba(X_test)[:, 1])
auc = roc_auc_score(y_test, rf.predict(X_test))
plot_roc_curve(fpr, tpr, auc, 'Random Forest (tuned)')

## 8. Summary

| Model | CV Accuracy | Test Accuracy | AUC |
|---|---|---|---|
| Logistic Regression | ~91% | 91.37% | — |
| Random Forest | ~91% | 91% | 0.73 |
| SVC | — | 90.9% | — |
| KNN | — | 90.0% | — |
| Decision Tree | — | 90.0% | — |

**Random Forest with tuned hyperparameters is selected** as the final model — strongest AUC and balanced precision/recall.